In [1]:
import sys, json, os; sys.path.append("..")
import pandas as pd, joblib
from sklearn.calibration import CalibratedClassifierCV
from src.train import make_lgbm

train = pd.read_csv("../data/processed/train.csv", index_col=0)
y = train.pop("SeriousDlqin2yrs"); X = train
best = json.load(open("../reports/best_lgbm_params.json"))

model = CalibratedClassifierCV(make_lgbm(**best), method="isotonic", cv=3).fit(X, y)

os.makedirs("../models", exist_ok=True)
joblib.dump({"model": model, "features": list(X.columns), "threshold": 1000 / 7000},
            "../models/credit_model.joblib")

# quick check: reload and predict 3 rows
saved = joblib.load("../models/credit_model.joblib")
print(saved["features"][:3], saved["model"].predict_proba(X.head(3))[:, 1].round(3))

['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse'] [0.003 0.034 0.024]
